# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and fields. All elements are referenced by their `@id` fields.

In [ ]:
# List all record set `@id`s and their corresponding fields/descriptions
record_set_ids = [rs['@id'] for rs in metadata._json_dict.get('recordSet', [])]

print('Record sets detected:')
for rs in metadata._json_dict.get('recordSet', []):
    print(f"@id: {rs['@id']} | name: {rs.get('name', 'N/A')} | description: {rs.get('description', 'N/A')}")
    print('  Fields:')
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        if isinstance(field, dict):
            print(f"    - @id: {field.get('@id', 'N/A')} | name: {field.get('name', 'N/A')}")
        else:
            print(f"    - @id: {field}")
    print('---')
# If no recordSet, let the user know
if not record_set_ids:
    print('No record sets found in the Croissant schema!')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use `@id`s from above.

In [ ]:
# If record sets are found, extract to DataFrame
dataframes = {}
if record_set_ids:
    for record_set in record_set_ids:
        print(f'Loading records for record set: {record_set}')
        try:
            records = list(dataset.records(record_set=record_set))
        except Exception as e:
            print(f'  Could not load record set {record_set}: {e}')
            continue
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[record_set] = df
            print(f'  Loaded {len(df)} records. Columns:')
            print(df.columns.tolist())
            print(df.head())
        else:
            print(f'  No records available for record set {record_set}')
else:
    print('No record sets found, cannot extract records.')

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering, normalizing, and grouping by attributes using `@id` references.

In [ ]:
# Explore records if available
import numpy as np

# Pick the first loaded record set for demonstration (update if a specific record set/column is needed)
if dataframes:
    record_set_example = list(dataframes.keys())[0]
    df = dataframes[record_set_example]

    # Show all columns
    print('Columns in this record set:')
    print(df.columns.tolist())

    # Try to guess a numeric field
    numeric_field_id = None
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field_id = col
            break
    if numeric_field_id is None:
        # Try to cast some columns to numeric (for CSVs with string numbers)
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
                if np.issubdtype(df[col].dtype, np.number):
                    numeric_field_id = col
                    break
            except Exception:
                continue

    if numeric_field_id is not None:
        print(f'Numeric field chosen for EDA: {numeric_field_id}')
        # Filter for values above mean
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):")
        print(filtered_df.head())

        # Normalizing
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a categorical field
        group_field = None
        for col in df.columns:
            if df[col].dtype == object and col != numeric_field_id:
                if df[col].nunique() < df.shape[0] // 2:
                    group_field = col
                    break
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean {numeric_field_id} by {group_field}:")
            print(grouped_df)
        else:
            print('No suitable categorical group field found for grouping.')
    else:
        print('No numeric field found for EDA!')
else:
    print('No records/dataframes available for EDA.')

## 5. Visualization
Visualize the selected numeric field's distribution and, if available, its grouping by a categorical attribute.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id is not None:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If a group field was found
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion
In this notebook, we've loaded and explored the dataset using its Croissant schema via `mlcroissant`. We discovered record sets, inspected available fields by their `@id`, extracted records into DataFrames, and performed a simple exploratory analysis including filtering and normalization of a numeric field, along with a visualization of its distribution.

Continue your analysis by selecting other record sets, exploring further fields referenced by their `@id`s, and applying more advanced processing or modeling as needed.